In [ ]:
# 📌 Google Colab 환경에서 실행하는 코드입니다.
# 필요한 라이브러리 설치
!pip install pykrx pandas numpy cvxopt

# --- 1️⃣ 라이브러리 불러오기 ---
import pandas as pd
import numpy as np
from pykrx import stock
from datetime import datetime
from cvxopt import matrix, solvers

# --- 2️⃣ ESG 데이터 로드 ---
esg_data = pd.read_csv("/content/final_esg_data.csv")

# 📌 티커 목록 로드 (중복 제거)
tickers = list(set(esg_data["ticker"].dropna().tolist()))
esg_scores = esg_data.set_index("ticker")["평균 점수"].dropna().to_dict()

# ✅ 2-1️⃣ `esg_scores`의 키 형식 변환 (".KS" 제거)
esg_scores = {t.replace(".KS", ""): v for t, v in esg_scores.items()}  # `.KS` 제거하여 비교

# --- 3️⃣ KRX에서 실제 존재하는 종목 리스트 가져오기 ---
krx_valid_tickers = stock.get_market_ticker_list(market="ALL")  # KOSPI & KOSDAQ 전체 종목

# 📌 KRX 종목코드 변환 (005930.KS → 005930)
krx_tickers = [t.replace(".KS", "") for t in tickers]

# 📌 KRX에서 실제 존재하는 종목만 필터링
valid_tickers = [t for t in krx_tickers if t in krx_valid_tickers]
print(f"✅ KRX에서 유효한 종목 개수: {len(valid_tickers)}")

# --- 4️⃣ KRX에서 주가 데이터 다운로드 ---
start_date = "2019-01-01"
end_date = "2023-12-31"

prices_file = "/content/stock_prices.csv"

try:
    prices = pd.read_csv(prices_file, index_col=0, parse_dates=True)
    print("✅ 기존 주가 데이터 로드 완료")
except FileNotFoundError:
    print("⚠️ 기존 데이터 없음 → KRX에서 다운로드 진행")
    prices = pd.DataFrame()

    for ticker in valid_tickers:
        try:
            stock_data = stock.get_market_ohlcv_by_date(start_date, end_date, ticker)["종가"]
            stock_data.name = ticker
            prices = pd.concat([prices, stock_data], axis=1)
        except Exception as e:
            print(f"❌ {ticker} 다운로드 실패: {e}")

    prices.ffill(inplace=True)
    prices.bfill(inplace=True)
    prices.to_csv(prices_file)  # ✅ 주가 데이터를 저장하여 이후 반복 다운로드 방지
    print("✅ 주가 데이터 저장 완료")

# --- 5️⃣ 수익률 및 공분산 행렬 계산 ---
# 📌 주가 데이터 컬럼명을 KRX 형식과 일치시킴
prices.columns = [col.replace(".KS", "") for col in prices.columns]

returns = prices.pct_change().dropna()

# 📌 `returns`에 포함되지 않은 종목 확인
tickers_in_returns = [t for t in valid_tickers if t in returns.columns]
missing_tickers = [t for t in valid_tickers if t not in returns.columns]

if missing_tickers:
    print(f"⚠️ `returns`에 없는 종목(삭제됨): {missing_tickers}")

# ✅ `returns` 데이터가 비어 있는 경우 예외 처리
if returns.empty:
    raise ValueError("❌ `returns` 데이터가 비어 있습니다. 주가 데이터 로딩을 확인하세요.")

# ✅ 기대수익률 `mu` 계산 (누락된 종목 제거 후 실행)
mu = returns[tickers_in_returns].mean() * 252  # 연간 기대수익률

if len(mu) == 0:
    raise ValueError("❌ `mu`가 비어 있습니다. 종목 코드 불일치 가능성 확인 필요!")

# --- 6️⃣ ESG 점수를 반영한 최적화 ---
# ✅ `tickers_in_returns`과 `esg_scores` 비교하여 존재하는 종목만 필터링
tickers_with_esg = [t for t in tickers_in_returns if t in esg_scores]

# ✅ 6-1️⃣ KeyError 방지를 위해 누락된 종목 확인
missing_esg_keys = [t for t in tickers_with_esg if t not in esg_scores]
if missing_esg_keys:
    print(f"⚠️ ESG 데이터에서 키를 찾을 수 없는 종목: {missing_esg_keys}")

# ✅ 6-2️⃣ `tickers_with_esg`가 존재하는지 확인
if len(tickers_with_esg) == 0:
    raise ValueError("❌ `tickers_with_esg`가 비어 있습니다. ESG 데이터와 주가 데이터 불일치 가능성 확인 필요!")

# ✅ 6-3️⃣ Q 벡터 생성 (ESG 점수 → 0~0.1로 정규화)
Q_values = np.array([esg_scores[t] for t in tickers_with_esg])

if len(Q_values) == 0:
    raise ValueError("❌ ESG 점수 데이터가 없습니다. 'final_esg_data.csv' 파일을 확인하세요.")

Q_min, Q_max = Q_values.min(), Q_values.max()
Q_scaled = (Q_values - Q_min) / (Q_max - Q_min) * 0.1
P = np.eye(len(Q_scaled))

# ✅ 7️⃣ `mu`도 동일한 종목 리스트로 필터링
mu = returns[tickers_with_esg].mean() * 252

# ✅ `mu`와 `Q_scaled` 크기 비교
print(f"🎯 유효한 기대수익률 데이터 개수: {len(mu)}")
print(f"🎯 Q 벡터 크기: {len(Q_scaled)}")

if len(mu) != len(Q_scaled):
    raise ValueError(f"❌ 기대수익률(mu) 크기 불일치: {len(mu)} vs {len(Q_scaled)}")

# ✅ 8️⃣ 최적화된 기대수익률 계산
tau = 0.05
adjusted_returns = tau * np.dot(P.T, Q_scaled) + (1 - tau) * mu.values[:len(Q_scaled)]
adjusted_returns = np.clip(adjusted_returns, -0.1, 0.2)

print("✅ 기대수익률 조정 완료!")

# --- 9️⃣ 포트폴리오 최적화 수행 (Quadratic Programming) ---
n = len(Q_scaled)
P_mat = matrix(sigma.loc[tickers_with_esg, tickers_with_esg].values)
q_mat = matrix(-adjusted_returns)
G_mat = matrix(np.vstack([-np.eye(n), np.eye(n)]))
h_mat = matrix(np.hstack([np.zeros(n), np.ones(n) * 0.1]))
A_mat = matrix(np.ones(n), (1, n))
b_mat = matrix(1.0)

sol = solvers.qp(P_mat, q_mat, G_mat, h_mat, A_mat, b_mat)
optimized_weights = np.array(sol["x"]).flatten()

# --- 🔟 최적화 결과 저장 ---
returns.to_csv("/content/returns_data.csv")
sigma.to_csv("/content/cov_matrix_data.csv")

optimized_weights_df = pd.DataFrame({"ticker": tickers_with_esg, "optimized_weight": optimized_weights})
optimized_weights_df.to_csv("/content/optimized_portfolio_weights.csv", index=False)

# --- ✅ 파일 생성 확인 ---
print("✅ 파일 생성 완료: returns_data.csv, cov_matrix_data.csv, optimized_portfolio_weights.csv")
